# 🎨 Blender Desktop on Kaggle — control from your Android phone

Fetches the setup from your **HF hub** (`amer21/blender-kaggle-hub`, includes the 377 MB Blender tarball), launches Blender + a public tunnel, and verifies it externally.

**Before running:** Kernel settings (⚙) → **Internet: On** (required), **GPU: T4** (recommended).

⚠️ **Important:** the tunnel only lives while the kernel is running. Run **Cell 5 (KEEP-ALIVE)** last — it blocks the kernel so Blender stays online. **Stop the kernel from the Kaggle UI when you're done** (frees GPU hours).

📱 Current phone URL is printed by Cells 2/4 and published to your private HF dataset `amer21/blender-backups` → `live-status.json`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — FETCH FROM HUGGING FACE & FULL SETUP  (~2-5 min)
#   Primary: your HF dataset (scripts + Blender tarball). Fallback: GitHub.
# ═══════════════════════════════════════════════════════════════════════════
import os, subprocess, shutil

WORK     = "/kaggle/working"
REPO_DIR = WORK + "/blender-kaggle"
HUB      = "amer21/blender-kaggle-hub"
GITHUB   = "https://github.com/amerameryou1-blip/blender-kaggle-desktop.git"

r = subprocess.run("curl -sI --max-time 10 https://huggingface.co | head -1", shell=True, capture_output=True, text=True)
if "200" not in r.stdout:
    raise SystemExit("⚠️  No internet. Kernel settings (⚙) → enable 'Internet' → Re-run this cell.")

os.system("pip install -q huggingface_hub")
shutil.rmtree(REPO_DIR, ignore_errors=True)
try:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HUB, repo_type="dataset", local_dir=REPO_DIR)
    print("✔ fetched setup from HF hub:", HUB)
except Exception as e:
    print("⚠ HF fetch failed, falling back to GitHub:", e)
    os.system(f"git clone --depth 1 {GITHUB} {REPO_DIR}")

env = dict(os.environ, WORKDIR=WORK)
subprocess.run(["bash", "setup.sh"], cwd=REPO_DIR, env=env, check=True)
subprocess.run(["bash", "launch.sh"], cwd=REPO_DIR, env=env, check=True)

url = open(os.path.join(WORK, "blender-env", "tunnel.url")).read().strip()
print("\n" + "=" * 64)
print("📱  OPEN THIS URL ON YOUR PHONE (Chrome or Edge):")
print()
print("   " + url)
print()
print("   Auto-connects, fits the screen. LANDSCAPE → fullscreen (⤢) → tap once.")
print("=" * 64)

In [ ]:
# CELL 2 — VERIFY EVERY COMPONENT (X / VNC handshake / noVNC / tunnel / Blender / GPU)
!bash /kaggle/working/blender-kaggle/test_components.sh

In [ ]:
# CELL 3 — GPU/CYCLES SMOKE TEST (headless; proves the T4 is used for rendering)
!bash /kaggle/working/blender-kaggle/run_render_test.sh
from IPython.display import Image, display
display(Image(filename="/kaggle/working/test-render.png"))

In [ ]:
# CELL 4 — (OPTIONAL) PERSISTENT BACKUPS TO HUGGING FACE + publish live URL
# Also needed if you want the live URL published to live-status.json.
import os, getpass
WORK = "/kaggle/working"
REPO = WORK + "/blender-kaggle"

tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    tok = getpass.getpass("Hugging Face token (or leave empty to skip): ").strip()

if tok:
    with open(WORK + "/.hf_token", "w") as f:
        f.write(tok)
    os.system("pip install -q huggingface_hub")
    os.system(f"WORKDIR={WORK} bash {REPO}/backup_to_hf.sh")
    os.system(f"nohup bash {REPO}/autosave_loop.sh >> {WORK}/blender-env/logs/autosave.log 2>&1 &")
    os.system(f"WORKDIR={WORK} python3 {REPO}/publish_status.py")
    print("✅ Backups every 10 min + live URL published to your HF.")
else:
    print("Skipped (no token). Blender still works; just no auto-backups / URL publishing.")

In [ ]:
# 🔗 GET CURRENT URL — run anytime the link stops working (self-healing)
!bash /kaggle/working/blender-kaggle/get_url.sh
!python3 /kaggle/working/blender-kaggle/publish_status.py || true

In [ ]:
# 🔒 CELL 5 — KEEP-ALIVE  (run LAST)
# Blocks the kernel so the tunnel + Blender stay online.
# ⛔ STOP THE KERNEL from the Kaggle UI when you're done (frees GPU hours).
import os, subprocess, time
REPO = "/kaggle/working/blender-kaggle"
print("🔒 Keep-alive running — Blender is online. URL re-verified every 2 min below.")
while True:
    try:
        r = subprocess.run(f"bash {REPO}/get_url.sh", shell=True, capture_output=True, text=True, timeout=200)
        lines = [l for l in r.stdout.strip().splitlines() if l]
        print(f"[live {time.strftime('%H:%M:%S')}] {lines[-1] if lines else '(no URL yet)'}", flush=True)
        subprocess.run(f"python3 {REPO}/publish_status.py", shell=True, timeout=120)
    except Exception as e:
        print("keep-alive hiccup:", e, flush=True)
    time.sleep(120)

In [ ]:
# 🛑 TEARDOWN — only if you're NOT keeping the session alive
!bash /kaggle/working/blender-kaggle/teardown.sh